# Car Price Prediction Model

This notebook builds a machine learning model to predict car selling prices based on various features including fuel type, transmission, mileage, and car age.

**Workflow:**
1. Data Loading & Exploration
2. Data Preprocessing & Feature Engineering
3. Model Training with Pipeline
4. Model Evaluation & Performance Metrics

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error as mse
import datetime

print("All libraries imported successfully")

All libraries imported successfully


In [2]:
## Step 1: Data Loading

# Download car price dataset from Kaggle
api = KaggleApi()
api.authenticate()
handle = 'vijayaadithyanvg/car-price-predictionused-cars'

api.dataset_download_files(handle, path='./', unzip=True)
print("Dataset downloaded successfully")

Dataset URL: https://www.kaggle.com/datasets/vijayaadithyanvg/car-price-predictionused-cars


Dataset downloaded successfully


In [3]:
# Load and preview the dataset
cars = pd.read_csv('car data.csv')
print(f"Dataset shape: {cars.shape}\n")
cars.head()

Dataset shape: (301, 9)



,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


## Step 2: Exploratory Data Analysis

In [4]:
# Display dataset information
print("Dataset Information:")
print(cars.info())
print('\n' + '='*70)
print(f"Dataset Shape: {cars.shape}")
print('='*70 + '\n')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Driven_kms     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Selling_type   301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB
None

Dataset Shape: (301, 9)



In [5]:
# Statistical summary
print("Statistical Summary:")
cars.describe().T

Statistical Summary:


,count,mean,std,min,25%,50%,75%,max
Year,301.0,2013.627907,2.891554,2003.00,2012.0,2014.0,2016.0,2018.0
Selling_Price,301.0,4.661296,5.082812,0.10,0.9,3.6,6.0,35.0
Present_Price,301.0,7.628472,8.642584,0.32,1.2,6.4,9.9,92.6
Driven_kms,301.0,36947.205980,38886.883882,500.00,15000.0,32000.0,48767.0,500000.0
Owner,301.0,0.043189,0.247915,0.00,0.0,0.0,0.0,3.0


In [6]:
# Feature engineering: Calculate car age and remove unnecessary columns
current_year = datetime.datetime.now().year
cars['Car_Age'] = current_year - cars['Year']
cars = cars.drop(['Year', 'Car_Name'], axis=1, inplace=False)

print("Car age feature created")
print("Year and Car_Name columns removed\n")
cars.head()

Car age feature created
Year and Car_Name columns removed



,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner,Car_Age
0,3.35,5.59,27000,Petrol,Dealer,Manual,0,12
1,4.75,9.54,43000,Diesel,Dealer,Manual,0,13
2,7.25,9.85,6900,Petrol,Dealer,Manual,0,9
3,2.85,4.15,5200,Petrol,Dealer,Manual,0,15
4,4.60,6.87,42450,Diesel,Dealer,Manual,0,12


In [8]:
# Check for missing values and duplicates
print("Missing Values:")
print(cars.isnull().sum())
print('\n' + '='*70 + '\n')

print(f"Duplicate rows: {cars.duplicated().sum()}")
cars = cars.drop_duplicates()
print("Data quality check complete")

Missing Values:
Selling_Price    0
Present_Price    0
Driven_kms       0
Fuel_Type        0
Selling_type     0
Transmission     0
Owner            0
Car_Age          0
dtype: int64


Duplicate rows: 2
Data quality check complete


## Step 3: Feature Preparation

### Identifying Feature Types

In [9]:
# Separate target and features
X = cars.drop(['Selling_Price'], axis=1)
y = cars['Selling_Price']

# Identify categorical and numeric features
categorical = ['Fuel_Type', 'Selling_type', 'Transmission']
numeric = [col for col in X.columns if col not in categorical]

print(f"Target variable: Selling_Price")
print(f"Categorical features: {categorical}")
print(f"Numeric features: {numeric}")

Target variable: Selling_Price
Categorical features: ['Fuel_Type', 'Selling_type', 'Transmission']
Numeric features: ['Present_Price', 'Driven_kms', 'Owner', 'Car_Age']


In [10]:
# Explore unique values in categorical columns
print("Unique values in categorical features:\n")
print(f"Fuel Types: {cars['Fuel_Type'].unique()}")
print(f"Selling Types: {cars['Selling_type'].unique()}")
print(f"Transmissions: {cars['Transmission'].unique()}")

Unique values in categorical features:

Fuel Types: ['Petrol' 'Diesel' 'CNG']
Selling Types: ['Dealer' 'Individual']
Transmissions: ['Manual' 'Automatic']


### Encoding Categorical Features

In [11]:
# Apply OneHotEncoding to categorical features
ohe = OneHotEncoder(sparse_output=False)

ct = make_column_transformer(
    (ohe, ['Fuel_Type']),
    (ohe, ['Selling_type']),
    (ohe, ['Transmission']),
    remainder='passthrough'
)

ct.set_output(transform='pandas')

cars_df = ct.fit_transform(cars)
print("✓ One-hot encoding applied to categorical features\n")
cars_df.head()

✓ One-hot encoding applied to categorical features



,onehotencoder-1__Fuel_Type_CNG,onehotencoder-1__Fuel_Type_Diesel,onehotencoder-1__Fuel_Type_Petrol,onehotencoder-2__Selling_type_Dealer,onehotencoder-2__Selling_type_Individual,onehotencoder-3__Transmission_Automatic,onehotencoder-3__Transmission_Manual,remainder__Selling_Price,remainder__Present_Price,remainder__Driven_kms,remainder__Owner,remainder__Car_Age
0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,3.35,5.59,27000,0,12
1,0.0,1.0,0.0,1.0,0.0,0.0,1.0,4.75,9.54,43000,0,13
2,0.0,0.0,1.0,1.0,0.0,0.0,1.0,7.25,9.85,6900,0,9
3,0.0,0.0,1.0,1.0,0.0,0.0,1.0,2.85,4.15,5200,0,15
4,0.0,1.0,0.0,1.0,0.0,0.0,1.0,4.60,6.87,42450,0,12


## Step 4: Train-Test Split

In [12]:
# Prepare features and target
X = cars_df.drop(['remainder__Selling_Price'], axis=1)
y = cars_df['remainder__Selling_Price']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (299, 11)
Target shape: (299,)


In [13]:
# Split data into training and testing sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=23)

print(f"✓ Training set size: {X_train.shape[0]} samples")
print(f"✓ Testing set size: {X_test.shape[0]} samples")
print(f"✓ Number of features: {X_train.shape[1]}")

✓ Training set size: 209 samples
✓ Testing set size: 90 samples
✓ Number of features: 11


## Step 5: Model Training

### Building ML Pipeline with Random Forest

In [ ]:
# Create ML pipeline: StandardScaler → Random Forest Regressor
scaler = StandardScaler()
rfr = RandomForestRegressor(n_estimators=200, random_state=35)

pipe = make_pipeline(scaler, rfr)

# Train the model
print("Training the Random Forest model...")
pipe.fit(X_train, y_train)
print("✓ Model training complete")

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('randomforestregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `cei

## Step 6: Model Evaluation & Performance Metrics

In [ ]:
# Generate predictions on test set
y_pred = pipe.predict(X_test)

# Display model accuracy scores
train_accuracy = pipe.score(X_train, y_train)
test_accuracy = pipe.score(X_test, y_test)

print("="*70)
print("MODEL PERFORMANCE SUMMARY")
print("="*70)
print(f"Training Accuracy (R² Score): {train_accuracy:.4f}")
print(f"Testing Accuracy (R² Score): {test_accuracy:.4f}")
print("="*70)

The training accuracy is:
0.9813911365746544
The testing accuracy is:
0.9323523820554196


In [ ]:
# Calculate detailed performance metrics
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse(y_test, y_pred))
mae = np.mean(np.abs(y_test - y_pred))

print("\nDetailed Performance Metrics:")
print(f"  R² Score: {r2:.4f}")
print(f"  Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"  Mean Absolute Error (MAE): ${mae:,.2f}")
print("\n" + "="*70)
print(f"✓ The model explains {r2*100:.2f}% of the variance in car prices")
print(f"✓ Average prediction error: ${rmse:,.2f}")
print("="*70)

The r2 score of the model is: 0.9323523820554196
The root mean square of the model is: 1.4938494294187499
